In [19]:
import pandas as pd

def split_csv_into_batches(file_path, batch_size=1000, required_columns=None):

    df = pd.read_csv(file_path)

    if df.index.name is not None:
        df.reset_index(drop=False, inplace=True)

    if required_columns:
        missing_columns = [col for col in required_columns if col not in df.columns]
        if missing_columns:
            raise ValueError(f"Missing required columns: {', '.join(missing_columns)}")

    # Ensure only the required columns are retained
    df = df[required_columns]
    print(f"df initial size: {df.shape}")

    def is_not_empty(value):
        return pd.notna(value) and str(value).strip() != ''

    for col in required_columns:
        df = df[df[col].apply(is_not_empty)]
        
    df = df.dropna(subset=["Byline"])

    print(f"df after dropping NaNs: {df.shape}")
    
    original_date_format = "%a %b %d %H:%M:%S %Z %Y"

    # Custom function to parse dates
    def parse_date(date_str):
        try:
            return pd.to_datetime(date_str, format=original_date_format, errors='coerce')
        except ValueError:
            return pd.to_datetime(date_str, infer_datetime_format=True, errors='coerce')

    # Apply date parsing function
    df['Publish Date'] = df['Publish Date'].apply(parse_date)

    df = df.dropna(subset=['Publish Date'])
    df = df.sort_values(by='Publish Date')

    def format_date(date):
        if pd.notna(date):
            return date.strftime(original_date_format)
        return ''

    df['Publish Date'] = df['Publish Date'].apply(format_date)
    
    total_rows = len(df)
    num_batches = (total_rows + batch_size - 1) // batch_size 
    # Split the DataFrame into batches and save each batch as a CSV
    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, total_rows)
        batch_df = df.iloc[start_idx:end_idx]
        batch_file_name = f"./../data_set/batch_{i+1}.csv"
        batch_df.to_csv(batch_file_name, index=False)
        print(f"Saved {batch_file_name} with {len(batch_df)} articles.")

    print("All batches have been saved successfully.")

In [20]:
file_path = "./../data_set/Articles_Nov_2020_March_2023.csv"
split_csv_into_batches(file_path, batch_size=1000, required_columns=["Byline", "Body", "Headline", "Publish Date", "Publisher", "Paths"])

df initial size: (12905, 6)
df after dropping NaNs: (12586, 6)
Saved ./../data_set/batch_1.csv with 1000 articles.
Saved ./../data_set/batch_2.csv with 1000 articles.
Saved ./../data_set/batch_3.csv with 1000 articles.
Saved ./../data_set/batch_4.csv with 1000 articles.
Saved ./../data_set/batch_5.csv with 1000 articles.
Saved ./../data_set/batch_6.csv with 914 articles.
All batches have been saved successfully.
